# CNN 

### Plan of action
1) Fashion Mnist on CNN with basic CNN 
2) optimize it and use Hyperparam
3) Then we will use Transfer Learning (using already made CNN models)

## CNN Theory

A special kind of NN for processing data that has a known grid like topology like time series data(1D) or images (2D)

### Difference B/W CNN and ANN
Convolution layer - performs convolution
in ANN we use matrix multiplication and in CNN we use convolution

In CNN we have 3 types of layers
- Convolution layers
- Pooling Layers 
- FC layers (fully connected layer)

### Inspired From
- Inspired by our own visual cortex

### Why not use ANN for image data 
- High Computation cost
- Overfitting 
- Loss of imp info like spacial arrangement of pixels

## Intuition of CNN
- It takes out some primitive features from the image which are known as edges
- Then it combines the features to form more complex edges
- Convolution - Filters - Extract features
    We move these filters on the image to find features then we send them to the next layer then next etc etc to form more complex features

## Uses 
- Image Classification
- Object Localisation - Find a object in image 
- Object Detection - Find all the objects in an image 
- Face Detection
- Image Segmentation - Divide the image into regions
- Super Resolution - Improve the resolution 
- Color photos - black white to color 
- pose estimation - finding current posture

### RoadMap
- Bio connection 
- Convolution - padding + stride 
- Pooling 
- CNN architecture 
- CNN vs ANN
- Dogs vs Cat classifier
- Data Augumentation
- Popular CNN architectures - LexNET , ALEXNEt , VggNET , RESNET , Transfer Learning

### CNN
- Works well in grid based data .
- identifies spacial patterns in the data.


### Our architecture
- Image input (1,28,28) (greyscale image)
- 2 layers of convolution and pooling layers 
- 1st layer has 32 filter , 0 padding , filter size 3*3 , pooling layer is of 2*2 and stride is 2
- 2nd layer has 64 filters, 0 padding , 0 padding , same for pooling layer
- Flatten layer
- FC layers - 128 neurons , 64 neurons 
- Output layer - 10 neurons

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch 
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn 
import torch.optim as optim
import matplotlib.pyplot as plt
import optuna

In [2]:
torch.manual_seed(42)
device = torch.device('cuda')

In [3]:
df = pd.read_csv(r"C:\Users\pieush\Desktop\Root\Work\Courses\Pytorch\ANN\fashion-mnist_train.csv")
df2 = pd.read_csv(r"C:\Users\pieush\Desktop\Root\Work\Courses\Pytorch\ANN\fashion-mnist_test.csv")

In [4]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values
# test dataset
X_v = df2.iloc[:,1:].values
y_v = df2.iloc[:,0].values

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_valid , y_valid = X_v , y_v

In [6]:
# scaling the features (0-1)
# advisable to scale
X_train = X_train/255.0
X_test = X_test/255.0
X_valid = X_valid/255.0
y_valid = y_valid/255.0

In [7]:
class CustomDataset(Dataset):

    def __init__(self,features,labels):

        self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28) #reshape into (batchsize,channel,width,height) 
        self.labels = torch.tensor(labels,dtype=torch.float32)
    
    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index] , self.labels[index]


In [8]:
train_dataset = CustomDataset(X_train , y_train)
test_dataset = CustomDataset(X_test,y_test)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32 , shuffle = True , pin_memory= True)
test_loader = DataLoader(test_dataset,batch_size=32 , shuffle= False , pin_memory= True)

In [10]:
# CNN archi creation

class MyNN(nn.Module):
    def __init__(self,input_features):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(input_features,32, kernel_size=3 , padding= 'same'),   #(in_channels , filters , kernel size , )
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size = 2 , stride = 2),
            
            nn.Conv2d(32,64,kernel_size=3 , padding= 'same'),   #(in_channels , filters , kernel size , )
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size = 2 , stride = 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p = 0.4),

            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p = 0.4),

            nn.Linear(64,10)
        )
    
    def forward(self , x):
        x = self.features(x)
        x = self.classifier(x)

        return x


In [11]:
learning_rate = 0.01
epochs = 100

In [12]:
model = MyNN(1)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr = learning_rate , weight_decay=1e-4)

In [25]:
for epoch in range(epochs):
        total_epoch_loss = 0
        for batch_features , batch_labels in train_loader:
            # move data to gpu
            batch_labels = batch_labels.type(torch.LongTensor) 
            batch_features,batch_labels = batch_features.to(device) , batch_labels.to(device)
            # forward pass
            outputs = model(batch_features) 
            # calculate loss
            """ print(outputs.shape,'\n')
            print(outputs[0].dtype,'\n')
            print(batch_labels.shape,'\n')
            print(batch_labels.dtype,'\n') """
            loss = criterion(outputs,batch_labels)   # outputs - Float , batch_labels -> long
            # back pass 
            optimizer.zero_grad()
            loss.backward()
            # update grads
            optimizer.step()
            total_epoch_loss = total_epoch_loss + loss.item()
        avg_loss = total_epoch_loss/len(train_loader)
        print(f'Epoch: {epoch + 1} , Loss: {avg_loss}') 

Epoch: 1 , Loss: 0.6404544817606608
Epoch: 2 , Loss: 0.3828358365992705
Epoch: 3 , Loss: 0.3212793045093616
Epoch: 4 , Loss: 0.2891033276990056
Epoch: 5 , Loss: 0.2628741712855796
Epoch: 6 , Loss: 0.24399455820086102
Epoch: 7 , Loss: 0.22292650466660657
Epoch: 8 , Loss: 0.20992106168468794
Epoch: 9 , Loss: 0.19623769102804361
Epoch: 10 , Loss: 0.1863971360878398
Epoch: 11 , Loss: 0.17739801443430284
Epoch: 12 , Loss: 0.16486819655634463
Epoch: 13 , Loss: 0.15830733390524984
Epoch: 14 , Loss: 0.14531856072104227
Epoch: 15 , Loss: 0.1400979876993224
Epoch: 16 , Loss: 0.13164866773085668
Epoch: 17 , Loss: 0.12685553793329746
Epoch: 18 , Loss: 0.11836197959600638
Epoch: 19 , Loss: 0.11347792537673376
Epoch: 20 , Loss: 0.10748189004471836
Epoch: 21 , Loss: 0.10051135559698256
Epoch: 22 , Loss: 0.09660567742570614
Epoch: 23 , Loss: 0.09264916363102384
Epoch: 24 , Loss: 0.08769493084703572
Epoch: 25 , Loss: 0.08407545821360933
Epoch: 26 , Loss: 0.08096712331543676
Epoch: 27 , Loss: 0.07810711

In [26]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
total = 0 
correct = 0

with torch.no_grad():

    for batch_features,batch_labels in test_loader:
        
        batch_features,batch_labels = batch_features.to(device) , batch_labels.to(device)

        outputs = model(batch_features)

        _,predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]

        correct = correct + (predicted==batch_labels).sum().item()
print(correct/total)

0.9255
